# Experiment Results Dashboard

Loads the latest `*_results_paper.json` files from each experiment suite and visualises key metrics.

In [ ]:
import json, os, glob
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib

matplotlib.rcParams.update({"font.size": 11, "figure.dpi": 120})

PAPER_DIR = Path(".")  # run from experiments/paper/

# Auto-discover the latest JSON result file in each suite
SUITES = {
    "Polynomial Learning": "polynomial_learning/poly_learning_results_paper.json",
    "Absolute Sum Minimization": "absolute_sum_minimization/absolute_sum_results_paper.json",
    "Function Learning": "function_learning/function_learning_results_paper.json",
    "Polynomial Solving": "polynomial_solving/polynomial_solving_results_paper.json",
}

data = {}
for name, rel in SUITES.items():
    p = PAPER_DIR / rel
    if p.exists():
        with open(p) as f:
            data[name] = json.load(f)
        print(f"✓ {name}: {p.name}  ({datetime.fromisoformat(data[name]['metadata']['timestamp']).strftime('%Y-%m-%d %H:%M')})")
    else:
        print(f"✗ {name}: not found at {p}")

print(f"\nLoaded {len(data)} / {len(SUITES)} suites")

## 1. Global Optimizer Rankings (across all configurations)

In [ ]:
fig, axes = plt.subplots(1, len(data), figsize=(5 * len(data), 5), sharey=False)
if len(data) == 1:
    axes = [axes]

for ax, (suite_name, d) in zip(axes, data.items()):
    gr = d["global_ranking"]
    # Sort by avg_rank
    optimizers = sorted(gr.keys(), key=lambda k: gr[k]["avg_rank"])
    ranks = [gr[o]["avg_rank"] for o in optimizers]
    colors = plt.cm.RdYlGn_r(np.linspace(0.15, 0.85, len(optimizers)))

    bars = ax.barh(optimizers, ranks, color=colors)
    ax.set_xlabel("Mean Rank (lower is better)")
    ax.set_title(suite_name, fontweight="bold")
    ax.invert_yaxis()
    for bar, r in zip(bars, ranks):
        ax.text(bar.get_width() + 0.1, bar.get_y() + bar.get_height() / 2,
                f"{r:.2f}", va="center", fontsize=9)

plt.tight_layout()
plt.show()

## 2. Per-Configuration Mean Rank Heatmap

In [ ]:
for suite_name, d in data.items():
    optimizer_order = d["metadata"]["optimizer_order"]
    configs = [exp["config"]["name"] for exp in d["experiments"]]

    matrix = np.full((len(configs), len(optimizer_order)), np.nan)
    for i, exp in enumerate(d["experiments"]):
        for j, opt in enumerate(optimizer_order):
            if opt in exp["aggregate"]:
                matrix[i, j] = exp["aggregate"][opt]["mean_rank"]

    fig, ax = plt.subplots(figsize=(max(8, len(optimizer_order) * 0.9), max(4, len(configs) * 0.45)))
    im = ax.imshow(matrix, cmap="RdYlGn_r", aspect="auto")
    ax.set_xticks(range(len(optimizer_order)))
    ax.set_xticklabels(optimizer_order, rotation=45, ha="right", fontsize=8)
    ax.set_yticks(range(len(configs)))
    ax.set_yticklabels(configs, fontsize=8)
    ax.set_title(f"{suite_name} — Mean Rank per Config", fontweight="bold")
    plt.colorbar(im, ax=ax, label="Mean Rank")

    # Annotate cells
    for i in range(matrix.shape[0]):
        for j in range(matrix.shape[1]):
            if not np.isnan(matrix[i, j]):
                ax.text(j, i, f"{matrix[i, j]:.1f}", ha="center", va="center", fontsize=7,
                        color="white" if matrix[i, j] > (np.nanmax(matrix) + np.nanmin(matrix)) / 2 else "black")

    plt.tight_layout()
    plt.show()

## 3. Mean Final Loss Comparison

In [ ]:
for suite_name, d in data.items():
    optimizer_order = d["metadata"]["optimizer_order"]
    experiments = d["experiments"]

    fig, axes = plt.subplots(1, len(experiments), figsize=(3.5 * len(experiments), 4), sharey=False)
    if len(experiments) == 1:
        axes = [axes]

    for ax, exp in zip(axes, experiments):
        opts = [o for o in optimizer_order if o in exp["aggregate"]]
        means = [exp["aggregate"][o]["mean_final_loss"] for o in opts]
        stds = [exp["aggregate"][o]["std_final_loss"] for o in opts]

        colors = plt.cm.tab10(np.arange(len(opts)) % 10)
        ax.bar(range(len(opts)), means, yerr=stds, color=colors, capsize=3)
        ax.set_xticks(range(len(opts)))
        ax.set_xticklabels(opts, rotation=60, ha="right", fontsize=7)
        ax.set_ylabel("Mean Final Loss")
        ax.set_title(exp["config"]["name"], fontsize=9)

    fig.suptitle(f"{suite_name} — Mean Final Loss", fontweight="bold", y=1.02)
    plt.tight_layout()
    plt.show()

## 4. Improvement Ratio

In [ ]:
for suite_name, d in data.items():
    optimizer_order = d["metadata"]["optimizer_order"]
    experiments = d["experiments"]

    # Aggregate improvement ratio across all configs
    imp = {o: [] for o in optimizer_order}
    for exp in experiments:
        for o in optimizer_order:
            if o in exp["aggregate"]:
                imp[o].append(exp["aggregate"][o]["mean_improvement_ratio"])

    opts = [o for o in optimizer_order if imp[o]]
    means = [np.mean(imp[o]) for o in opts]
    stds = [np.std(imp[o]) for o in opts]

    fig, ax = plt.subplots(figsize=(8, 4))
    idx = np.argsort(means)[::-1]
    sorted_opts = [opts[i] for i in idx]
    sorted_means = [means[i] for i in idx]
    sorted_stds = [stds[i] for i in idx]

    bars = ax.barh(sorted_opts, sorted_means, xerr=sorted_stds, capsize=3,
                   color=plt.cm.viridis(np.linspace(0.2, 0.8, len(sorted_opts))))
    ax.set_xlabel("Mean Improvement Ratio (higher is better)")
    ax.set_title(f"{suite_name} — Improvement Ratio (avg across configs)", fontweight="bold")
    ax.invert_yaxis()
    plt.tight_layout()
    plt.show()

## 5. Timing & Function Evaluations

In [ ]:
for suite_name, d in data.items():
    optimizer_order = d["metadata"]["optimizer_order"]
    experiments = d["experiments"]

    # Aggregate timing and evals
    times = {o: [] for o in optimizer_order}
    evals = {o: [] for o in optimizer_order}
    for exp in experiments:
        for o in optimizer_order:
            if o in exp["aggregate"]:
                times[o].append(exp["aggregate"][o]["mean_time"])
                evals[o].append(exp["aggregate"][o]["mean_total_evals"])

    opts = [o for o in optimizer_order if times[o]]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

    # Time
    mean_t = [np.mean(times[o]) for o in opts]
    ax1.barh(opts, mean_t, color="steelblue")
    ax1.set_xlabel("Mean Time (s)")
    ax1.set_title("Wall-Clock Time")
    ax1.invert_yaxis()

    # Evals
    mean_e = [np.mean(evals[o]) for o in opts]
    ax2.barh(opts, mean_e, color="darkorange")
    ax2.set_xlabel("Mean Function Evaluations")
    ax2.set_title("Function Evaluations")
    ax2.invert_yaxis()

    fig.suptitle(f"{suite_name} — Efficiency", fontweight="bold")
    plt.tight_layout()
    plt.show()

## 6. Loss Trajectories (sample-level)

In [ ]:
# Show loss trajectories for the first experiment config in each suite
for suite_name, d in data.items():
    exp = d["experiments"][0]
    optimizer_order = d["metadata"]["optimizer_order"]
    config_name = exp["config"]["name"]

    fig, ax = plt.subplots(figsize=(8, 4))

    for opt in optimizer_order:
        # Collect all loss trajectories for this optimizer
        trajectories = []
        for sample in exp["samples"]:
            if opt in sample["optimizers"] and "losses" in sample["optimizers"][opt]:
                trajectories.append(sample["optimizers"][opt]["losses"])

        if not trajectories:
            continue

        # Compute mean trajectory (pad shorter ones)
        max_len = max(len(t) for t in trajectories)
        padded = np.full((len(trajectories), max_len), np.nan)
        for i, t in enumerate(trajectories):
            padded[i, :len(t)] = t

        mean_traj = np.nanmean(padded, axis=0)
        ax.plot(mean_traj, label=opt, alpha=0.8)

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss")
    ax.set_title(f"{suite_name} — Loss Trajectories ({config_name})", fontweight="bold")
    ax.legend(fontsize=7, ncol=2, loc="upper right")
    plt.tight_layout()
    plt.show()

## 7. Summary Table

In [ ]:
# Build a combined summary DataFrame
rows = []
for suite_name, d in data.items():
    gr = d["global_ranking"]
    for opt, stats in gr.items():
        # Also compute mean improvement ratio and mean time across configs
        imp_ratios, mean_times, mean_evals = [], [], []
        for exp in d["experiments"]:
            if opt in exp["aggregate"]:
                agg = exp["aggregate"][opt]
                imp_ratios.append(agg["mean_improvement_ratio"])
                mean_times.append(agg["mean_time"])
                mean_evals.append(agg["mean_total_evals"])
        rows.append({
            "Suite": suite_name,
            "Optimizer": opt,
            "Avg Rank": round(stats["avg_rank"], 2),
            "Configs": stats["n_configs"],
            "Mean Impr. Ratio": round(np.mean(imp_ratios), 3) if imp_ratios else None,
            "Mean Time (s)": round(np.mean(mean_times), 4) if mean_times else None,
            "Mean Evals": round(np.mean(mean_evals), 1) if mean_evals else None,
        })

df = pd.DataFrame(rows)
df = df.sort_values(["Suite", "Avg Rank"])

# Style: highlight best rank per suite
def highlight_best(s):
    is_best = s == s.min()
    return ["font-weight: bold; background-color: #d4edda" if v else "" for v in is_best]

styled = (df.style
    .apply(highlight_best, subset=["Avg Rank"])
    .format({"Mean Impr. Ratio": "{:.3f}", "Mean Time (s)": "{:.4f}", "Avg Rank": "{:.2f}"})
    .hide(axis="index"))

styled